In [ ]:
import pandas as pd

catalogue = pd.read_csv("../catalogue/catalogue_base.csv")
catalogue.head(1)

,tconst,titre_international,titre_affiche,annee_sortie,duree_min,genres,note_moyenne,nombre_votes,original_language,production_countries,production_companies,overview,poster_path,affiche_url,film_francais
0,tt0006864,Intolerance,Intolérance,1916,210,"Drama,History",7.6,18277,en,United States of America,"Triangle Film Corporation, Wark Producing Corp.","The story of a poor young woman, separated by ...",/lJCabs4TMOsKfnl0BmA5Hf16Ezb.jpg,https://image.tmdb.org/t/p/w500/lJCabs4TMOsKfn...,False


In [ ]:
# title.principals par morceaux : nos films + acteurs, actrices, réalisateurs
mes_films = set(catalogue["tconst"])
morceaux = []
lien = "https://datasets.imdbws.com/title.principals.tsv.gz"

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["tconst", "ordering", "nconst", "category"],
                           chunksize=2_000_000):
    est_a_moi = morceau["tconst"].isin(mes_films)
    bon_role = morceau["category"].isin(["actor", "actress", "director"])
    morceaux.append(morceau[est_a_moi & bon_role])

principals = pd.concat(morceaux)
print(principals.shape)

In [ ]:
# name.basics par morceaux : seulement les personnes de nos films
mes_personnes = set(principals["nconst"])
morceaux = []
lien = "https://datasets.imdbws.com/name.basics.tsv.gz"

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["nconst", "primaryName"],
                           chunksize=2_000_000):
    morceaux.append(morceau[morceau["nconst"].isin(mes_personnes)])

noms = pd.concat(morceaux)

# Ajouter les noms aux rôles
principals = pd.merge(principals, noms, on="nconst", how="left")
principals = principals.dropna(subset="primaryName")
principals = principals.drop_duplicates(subset=["tconst", "nconst"])
principals = principals.sort_values(["tconst", "ordering"])

In [ ]:
# Une ligne par film : les X premiers acteurs, et les réalisateurs
acteurs = principals[principals["category"].isin(["actor", "actress"])]
acteurs = acteurs.groupby("tconst").head(3)          # combien d'acteurs principaux ?
acteurs = acteurs.groupby("tconst")["primaryName"].agg(", ".join).reset_index()
acteurs = acteurs.rename(columns={"primaryName": "acteurs"})

realisateurs = principals[principals["category"] == "director"]
realisateurs = realisateurs.groupby("tconst")["primaryName"].agg(", ".join).reset_index()
realisateurs = realisateurs.rename(columns={"primaryName": "realisateurs"})

# Jointure avec le catalogue
catalogue = pd.merge(catalogue, acteurs, on="tconst", how="left")
catalogue = pd.merge(catalogue, realisateurs, on="tconst", how="left")
print(len(catalogue))
print(catalogue[["acteurs", "realisateurs"]].isna().sum())
catalogue[["titre_affiche", "acteurs", "realisateurs"]].head()

In [ ]:
catalogue["genres"].str.split(",").explode().value_counts()

In [ ]:
#Création du dictionnaire des emotions

emotions = {
    "rire":       ["Comedy", "Animation", "Family"],
    "frissonner": ["Horror", "Thriller", "Mystery", "Crime", "Film-Noir"],
    "emouvoir":   ["Romance", "Music", "Musical"],
    "evader":     ["Adventure", "Fantasy", "Sci-Fi", "Animation", "Family"],
    "reflechir":  ["Documentary", "Biography", "History", "News"],
    "vibrer":     ["Action", "War", "Sport", "Western"],
}

In [ ]:
# Colonne vrai/faux par emotion

for emotion, genres in emotions.items():
    catalogue[f"emo_{emotion}"] = catalogue["genres"].str.contains("|".join(genres), na=False)

In [ ]:
# règle speciale pour le genre Drama qui represente 71% u catalogue

drame_seul = catalogue["genres"] == "Drama"
catalogue["emo_emouvoir"] = catalogue["emo_emouvoir"] | drame_seul
print(drame_seul.sum())

In [ ]:
# Verification

colonnes_emo = [c for c in catalogue.columns if c.startswith("emo_")]
print(catalogue[colonnes_emo].sum())                        # films par émotion
print((catalogue[colonnes_emo].sum(axis=1) == 0).sum())     # films sans émotion

In [ ]:
# Sauvegarde
catalogue.to_csv("../catalogue/catalogue.csv", index=False)